In [1]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [2]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [3]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 19:09:37, wtch_dt_end:2026-07-21 19:09:37


In [4]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [5]:
@file:DependsOn("org.json:json:20250107")

In [6]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [20]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2167,1091,0,0.260000,61,5.199049,2.514356,0.250000,3.949333,5.520000,6.439000,19.907000
rtmWqChpla,Comparable<*>,2167,1327,0,,182,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2167,1,0,,2167,null,null,,,,,
rtmWqWtchStaCd,String,2167,14,0,SEA1005,182,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2167,2167,0,1,1,1084.000000,625.703338,1,542.166667,1084.000000,1625.833333,2167
rtmWqTu,Int,2167,160,0,5,198,25.550531,33.890829,0,5.000000,11.000000,31.833333,232
ph,Double,2167,143,0,7.500000,53,7.678440,0.300318,7.020000,7.480000,7.630000,7.920000,9.080000
rtmWqSlnty,Number,2167,2006,0,32.705002,4,21.834439,10.285551,0.020000,13.946000,26.650000,29.558001,34.032001
rtmWqCndctv,Float,2167,2079,0,44.272999,3,34.090563,15.385448,0.046000,23.212500,39.870998,45.027000,54.451000
rtmWqWtchDtlDt,String,2167,192,0,2026-07-20 19:10:00.0,14,null,null,2026-07-20 19:10:00.0,2026-07-21 01:10:00.0,2026-07-21 07:10:00.0,2026-07-21 13:10:00.0,2026-07-21 19:00:00.0


In [47]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) "0" else String.format("%.2f", value.toFloat())

}.convert { rtmWqDoxn and  rtmWqTu and rtmWqSlnty and rtmWqCndctv and rtmWtchWtem }.with { String.format("%.2f", it.toFloat()) }


df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,String,2167,671,0,0.26,61,null,null,0.25,3.75,5.46,6.37,9.90
rtmWqChpla,String,2167,950,0,0,182,null,null,0,1.30,17.47,4.13,9.92
rtmWqWtchStaCd,String,2167,14,0,SEA1005,182,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2167,2167,0,1,1,1084.000000,625.703338,1,542.166667,1084.000000,1625.833333,2167
rtmWqTu,String,2167,160,0,5.00,198,null,null,0.00,14.00,33.00,59.00,99.00
ph,Double,2167,143,0,7.500000,53,7.678440,0.300318,7.020000,7.480000,7.630000,7.920000,9.080000
rtmWqSlnty,String,2167,1242,0,32.74,10,null,null,0.02,19.58,27.13,30.36,9.95
rtmWqCndctv,String,2167,1492,0,44.44,9,null,null,0.05,27.58,4.92,45.18,9.86
rtmWqWtchDtlDt,LocalDateTime,2167,192,0,2026-07-20T19:10,14,null,null,2026-07-20T19:10,2026-07-21T01:10,2026-07-21T07:10,2026-07-21T13:10,2026-07-21T19:00
rtmWtchWtem,String,2167,762,0,26.99,13,null,null,19.74,24.98,26.46,28.19,30.66


In [48]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
용존산소,String,2167,671,0,0.26,61,null,null,0.25,3.75,5.46,6.37,9.90
클로로필,String,2167,950,0,0,182,null,null,0,1.30,17.47,4.13,9.92
관측정점코드,String,2167,14,0,SEA1005,182,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
순번,Int,2167,2167,0,1,1,1084.000000,625.703338,1,542.166667,1084.000000,1625.833333,2167
탁도,String,2167,160,0,5.00,198,null,null,0.00,14.00,33.00,59.00,99.00
수소이온농도,Double,2167,143,0,7.500000,53,7.678440,0.300318,7.020000,7.480000,7.630000,7.920000,9.080000
염분,String,2167,1242,0,32.74,10,null,null,0.02,19.58,27.13,30.36,9.95
전기전도도,String,2167,1492,0,44.44,9,null,null,0.05,27.58,4.92,45.18,9.86
일시,LocalDateTime,2167,192,0,2026-07-20T19:10,14,null,null,2026-07-20T19:10,2026-07-21T01:10,2026-07-21T07:10,2026-07-21T13:10,2026-07-21T19:00
수온,String,2167,762,0,26.99,13,null,null,19.74,24.98,26.46,28.19,30.66


In [49]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1,2026-07-20T19:10,1.58,0,SEA1005,14.00,7.290000,0.51,1.04,27.75
2,2026-07-20T19:10,7.48,2.56,NEP1002,9.00,7.970000,1.04,2.04,29.12
3,2026-07-20T19:10,5.59,7.58,SEA5003,13.00,7.570000,32.63,49.86,25.00
4,2026-07-20T19:10,0.95,2.03,SEA5002,55.00,7.360000,15.35,25.34,29.98
5,2026-07-20T19:10,6.04,3.02,NEP2001,7.00,7.830000,14.43,23.84,26.47


In [51]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .convert{수온}.toDouble()
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온 °C"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="tyhBn4" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("tyhBn4");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E